In [14]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.01
grid_min_x = 0.
grid_max_x = 8. * np.pi
grid_min_z = 0.
grid_max_z = 3. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Calculate invariants
Gamma = np.matmul(grads, np.transpose(grads, (0, 1, 3, 2)))
s1 = np.sqrt(Gamma[:, :, 0, 0])
s2 = np.sqrt(Gamma[:, :, 0, 0] * Gamma[:, :, 1, 1] - Gamma[:, :, 0, 1]**2)
s3 = np.sqrt(np.linalg.det(Gamma))

# Flatten invariants
S1 = s1.flatten()
S2 = s2.flatten()
S3 = s3.flatten()

# Calculate invariant ratios
inv = np.zeros((5, num), dtype=np.float32)
inv[0, :] = S3 / (S1**3)
inv[1, :] = S2 / (S1**2)
inv[2, :] = (S3**2) / (S2**3)
inv[3, :] = (S3**2) / (S1**2 * S2**2)
inv[4, :] = (S1 * S3) / (S2**2)

# Save results
np.save('AtA-invar-all-channel-001.npy', inv)

# Save velocities and gradients
np.save('velocities.npy', veloc)
np.save('gradients.npy', grads)

# Part 4: Code to read from file instead of database
def process_from_file():
    veloc = np.load('velocities.npy')
    grads = np.load('gradients.npy')

    # Calculate invariants
    Gamma = np.matmul(grads, np.transpose(grads, (0, 1, 3, 2)))
    s1 = np.sqrt(Gamma[:, :, 0, 0])
    s2 = np.sqrt(Gamma[:, :, 0, 0] * Gamma[:, :, 1, 1] - Gamma[:, :, 0, 1]**2)
    s3 = np.sqrt(np.linalg.det(Gamma))

    # Flatten invariants
    S1 = s1.flatten()
    S2 = s2.flatten()
    S3 = s3.flatten()

    # Calculate invariant ratios
    inv = np.zeros((5, len(S1)), dtype=np.float32)
    inv[0, :] = S3 / (S1**3)
    inv[1, :] = S2 / (S1**2)
    inv[2, :] = (S3**2) / (S2**3)
    inv[3, :] = (S3**2) / (S1**2 * S2**2)
    inv[4, :] = (S1 *S3) / (S2**2)

    # Save results
    np.save('AtA-invar-all-channel-001-from-file.npy', inv)

# Uncomment the following line to run Part 4
# process_from_file()

/var/folders/pl/8kjsy7p54kjd1k8dbsg9yz840000gn/T/ipykernel_94839/3064072932.py:87: RuntimeWarning: invalid value encountered in sqrt
  s3 = np.sqrt(np.linalg.det(Gamma))
/var/folders/pl/8kjsy7p54kjd1k8dbsg9yz840000gn/T/ipykernel_94839/3064072932.py:96: RuntimeWarning: invalid value encountered in divide
  inv[0, :] = S3 / (S1**3)


ValueError: could not broadcast input array from shape (2371000,) into shape (2370702,)

In [3]:
import numpy as np

def process_from_file():
    veloc = np.load('velocities.npy')
    grads = np.load('gradients.npy')

    # Check dimensions of grads
    if grads.ndim != 4:
        raise ValueError("grads array must be 4-dimensional")

    # Calculate invariants
    Gamma = np.matmul(grads, np.transpose(grads, (0, 1, 3, 2)))
     
    # Compute eigenvalues of each matrix in Gamma
    eigenvalues = np.linalg.eigvals(Gamma)
    
    # Check if all eigenvalues are positive
    positive_definite_indices = np.all(eigenvalues > 0, axis=-1)
    Gamma = Gamma[positive_definite_indices]

    # Calculate s1, s2, s3 with additional checks
    s1 = np.sqrt(np.maximum(Gamma[:, 0, 0], 0))
    s2 = np.sqrt(np.maximum(Gamma[:, 0, 0] * Gamma[:, 1, 1] - Gamma[:, 0, 1]**2, 0))
    s3 = np.sqrt(np.maximum(np.linalg.det(Gamma), 0))

    # Combine all exclusion criteria
    non_zero_indices = (s1 > 0) & (s2 > 0) & (s3 > 0)

    # Apply combined filter to all relevant arrays
    s1 = s1[non_zero_indices]
    s2 = s2[non_zero_indices]
    s3 = s3[non_zero_indices]

    # Flatten invariants (не нужно, так как уже одномерные)
    S1 = s1
    S2 = s2
    S3 = s3

    # Calculate invariant ratios
    inv = np.zeros((5, len(S1)), dtype=np.float32)
    inv[0, :] = S3 / (S1**3)
    inv[1, :] = S2 / (S1**2)
    inv[2, :] = (S3**2) / (S2**3)
    inv[3, :] = (S3**2) / (S1**2 * S2**2)
    inv[4, :] = (S1 * S3) / (S2**2)

    # Save results
    np.save('AtA-invar-all-channel-001-from-file.npy', inv)

process_from_file()


In [4]:
InvLoadChanGrid0=np.load('AtA-invar-all-channel-001-from-file.npy')

In [5]:
len(InvLoadChanGrid0[0])

2370288

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.2

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-02.npy', veloc)
np.save('gradients-02.npy', grads)

In [ ]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.3

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-03.npy', veloc)
np.save('gradients-03.npy', grads)

In [4]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.4

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-04.npy', veloc)
np.save('gradients-04.npy', grads)

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.5

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-05.npy', veloc)
np.save('gradients-05.npy', grads)

In [ ]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.6

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-06.npy', veloc)
np.save('gradients-06.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.7

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-07.npy', veloc)
np.save('gradients-07.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.8

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-08.npy', veloc)
np.save('gradients-08.npy', grads)

: 

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.9

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-09.npy', veloc)
np.save('gradients-09.npy', grads)

In [1]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()


# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.96

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-096.npy', veloc)
np.save('gradients-096.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.98

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-098.npy', veloc)
np.save('gradients-098.npy', grads)

In [4]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.99

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-099.npy', veloc)
np.save('gradients-099.npy', grads)

: 

In [1]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.995

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0995.npy', veloc)
np.save('gradients-0995.npy', grads)

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.985

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0985.npy', veloc)
np.save('gradients-0985.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.988

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0988.npy', veloc)
np.save('gradients-0988.npy', grads)

In [4]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.992

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0992.npy', veloc)
np.save('gradients-0992.npy', grads)

In [5]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.997

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0997.npy', veloc)
np.save('gradients-0997.npy', grads)

In [6]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.998

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0998.npy', veloc)
np.save('gradients-0998.npy', grads)

: 

In [1]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.999

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0999.npy', veloc)
np.save('gradients-0999.npy', grads)

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.996

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0996.npy', veloc)
np.save('gradients-0996.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.994

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0994.npy', veloc)
np.save('gradients-0994.npy', grads)

In [4]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.993

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-0993.npy', veloc)
np.save('gradients-0993.npy', grads)

: 

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.94

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-094.npy', veloc)
np.save('gradients-094.npy', grads)

In [ ]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.92

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-092.npy', veloc)
np.save('gradients-092.npy', grads)

In [10]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# load shared library
lTDB = pyJHTDB.libJHTDB()
#initialize webservices
lTDB.initialize()

#Add token
auth_token  = "edu.jhu.pha.turbulence.testing-201311"  #Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np. pi
grid_max_z = 2. * np.pi


# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# Fixed value for the 1-coordinate (y)
fixed_y_value = 0.85

# Generate grid points
x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)

for nn in range(num_cycles):
    r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
    points = np.zeros((num_points, 3), dtype=np.float32)
    for j in range(num_points):
        x_index = (nn * num_points + j) % num_points_x
        z_index = (nn * num_points + j) // num_points_x
        if z_index >= num_points_z:
            break
        points[j, 0] = x_values[x_index]
        points[j, 1] = fixed_y_value 
        points[j, 2] = z_values[z_index]
        for i in range(3):
            for k in range(num_ints):
                for m in range(num_shifts):
                    r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
    v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
    veloc[nn, :len(v)] = v

# Calculate gradients
grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
for i in range(3):
    grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

# Save velocities and gradients
np.save('velocities-085.npy', veloc)
np.save('gradients-085.npy', grads)

In [1]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# Load shared library
lTDB = pyJHTDB.libJHTDB()
# Initialize webservices
lTDB.initialize()

# Add token
auth_token = "edu.jhu.pha.turbulence.testing-201311"  # Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np.pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# List of y values to iterate over
y_values = [0.91, 0.92, 0.93, 0.95]

# Generate grid points and process for each y value
for fixed_y_value in y_values:
    # Generate grid points
    x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
    z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)
    
    for nn in range(num_cycles):
        r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
        points = np.zeros((num_points, 3), dtype=np.float32)
        for j in range(num_points):
            x_index = (nn * num_points + j) % num_points_x
            z_index = (nn * num_points + j) // num_points_x
            if z_index >= num_points_z:
                break
            points[j, 0] = x_values[x_index]
            points[j, 1] = fixed_y_value 
            points[j, 2] = z_values[z_index]
            for i in range(3):
                for k in range(num_ints):
                    for m in range(num_shifts):
                        r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
        v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
        veloc[nn, :len(v)] = v

    # Calculate gradients
    grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
    for i in range(3):
        grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

    # Save velocities and gradients with the corresponding y value in the filename
    np.save(f'velocities-{fixed_y_value:.3f}.npy', veloc)
    np.save(f'gradients-{fixed_y_value:.3f}.npy', grads)

In [2]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# Load shared library
lTDB = pyJHTDB.libJHTDB()
# Initialize webservices
lTDB.initialize()

# Add token
auth_token = "edu.jhu.pha.turbulence.testing-201311"  # Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np.pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# List of y values to iterate over
y_values = [0.905, 0.915, 0.97, 0.975]

# Generate grid points and process for each y value
for fixed_y_value in y_values:
    # Generate grid points
    x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
    z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)
    
    for nn in range(num_cycles):
        r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
        points = np.zeros((num_points, 3), dtype=np.float32)
        for j in range(num_points):
            x_index = (nn * num_points + j) % num_points_x
            z_index = (nn * num_points + j) // num_points_x
            if z_index >= num_points_z:
                break
            points[j, 0] = x_values[x_index]
            points[j, 1] = fixed_y_value 
            points[j, 2] = z_values[z_index]
            for i in range(3):
                for k in range(num_ints):
                    for m in range(num_shifts):
                        r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
        v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
        veloc[nn, :len(v)] = v

    # Calculate gradients
    grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
    for i in range(3):
        grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

    # Save velocities and gradients with the corresponding y value in the filename
    np.save(f'velocities-{fixed_y_value:.3f}.npy', veloc)
    np.save(f'gradients-{fixed_y_value:.3f}.npy', grads)

In [3]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# Load shared library
lTDB = pyJHTDB.libJHTDB()
# Initialize webservices
lTDB.initialize()

# Add token
auth_token = "edu.jhu.pha.turbulence.testing-201311"  # Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np.pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# List of y values to iterate over
y_values = [0.925, 0.935]

# Generate grid points and process for each y value
for fixed_y_value in y_values:
    # Generate grid points
    x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
    z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)
    
    for nn in range(num_cycles):
        r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
        points = np.zeros((num_points, 3), dtype=np.float32)
        for j in range(num_points):
            x_index = (nn * num_points + j) % num_points_x
            z_index = (nn * num_points + j) // num_points_x
            if z_index >= num_points_z:
                break
            points[j, 0] = x_values[x_index]
            points[j, 1] = fixed_y_value 
            points[j, 2] = z_values[z_index]
            for i in range(3):
                for k in range(num_ints):
                    for m in range(num_shifts):
                        r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
        v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
        veloc[nn, :len(v)] = v

    # Calculate gradients
    grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
    for i in range(3):
        grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

    # Save velocities and gradients with the corresponding y value in the filename
    np.save(f'velocities-{fixed_y_value:.3f}.npy', veloc)
    np.save(f'gradients-{fixed_y_value:.3f}.npy', grads)

In [5]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# Load shared library
lTDB = pyJHTDB.libJHTDB()
# Initialize webservices
lTDB.initialize()

# Add token
auth_token = "edu.jhu.pha.turbulence.testing-201311"  # Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np.pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# List of y values to iterate over
y_values = [0.991, 0.989]

# Generate grid points and process for each y value
for fixed_y_value in y_values:
    # Generate grid points
    x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
    z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)
    
    for nn in range(num_cycles):
        r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
        points = np.zeros((num_points, 3), dtype=np.float32)
        for j in range(num_points):
            x_index = (nn * num_points + j) % num_points_x
            z_index = (nn * num_points + j) // num_points_x
            if z_index >= num_points_z:
                break
            points[j, 0] = x_values[x_index]
            points[j, 1] = fixed_y_value 
            points[j, 2] = z_values[z_index]
            for i in range(3):
                for k in range(num_ints):
                    for m in range(num_shifts):
                        r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
        v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
        veloc[nn, :len(v)] = v

    # Calculate gradients
    grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
    for i in range(3):
        grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

    # Save velocities and gradients with the corresponding y value in the filename
    np.save(f'velocities-{fixed_y_value:.3f}.npy', veloc)
    np.save(f'gradients-{fixed_y_value:.3f}.npy', grads)

In [6]:
import numpy as np
import time as tt
import matplotlib.pyplot as plt
import pyJHTDB

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = "serif"

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 100000

time = 3.3

# Load shared library
lTDB = pyJHTDB.libJHTDB()
# Initialize webservices
lTDB.initialize()

# Add token
auth_token = "edu.jhu.pha.turbulence.testing-201311"  # Replace with your own token here
lTDB.add_token(auth_token)

# Grid parameters
grid_spacing = 0.02
grid_min_x = 2. * np.pi
grid_max_x = 6. * np.pi
grid_min_z = np.pi
grid_max_z = 2. * np.pi

# Calculate the number of points in each dimension
num_points_x = int((grid_max_x - grid_min_x) / grid_spacing) + 1
num_points_z = int((grid_max_z - grid_min_z) / grid_spacing) + 1

# Total number of points
num = num_points_x * num_points_z

# Constants
num_points = 500  # Number of points per cycle
shift = np.array([-.0001, .0001])
num_shifts = len(shift)
num_ints = 3
de = np.identity(3)

# Calculate the number of cycles
num_cycles = (num + num_points - 1) // num_points

# Pre-allocate arrays
veloc = np.zeros((num_cycles, num_points, num_ints, num_shifts, 3), dtype=np.float32)
points = np.zeros((num_points, 3), dtype=np.float32)
r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)

# List of y values to iterate over
y_values = [0.975, 0.965]

# Generate grid points and process for each y value
for fixed_y_value in y_values:
    # Generate grid points
    x_values = np.linspace(grid_min_x, grid_max_x, num_points_x)
    z_values = np.linspace(grid_min_z, grid_max_z, num_points_z)
    
    for nn in range(num_cycles):
        r = np.zeros((num_points, num_ints, num_shifts, 3), dtype=np.float32)
        points = np.zeros((num_points, 3), dtype=np.float32)
        for j in range(num_points):
            x_index = (nn * num_points + j) % num_points_x
            z_index = (nn * num_points + j) // num_points_x
            if z_index >= num_points_z:
                break
            points[j, 0] = x_values[x_index]
            points[j, 1] = fixed_y_value 
            points[j, 2] = z_values[z_index]
            for i in range(3):
                for k in range(num_ints):
                    for m in range(num_shifts):
                        r[j, k, m, i] = points[j, i] + shift[m] * de[i, k]
        v = lTDB.getData(time, r, data_set='channel', sinterp=4, getFunction='getVelocity')
        veloc[nn, :len(v)] = v

    # Calculate gradients
    grads = np.zeros((num_cycles, num_points, 3, 3), dtype=np.float32)
    for i in range(3):
        grads[:, :, i, :] = (veloc[:, :, i, 1, :] - veloc[:, :, i, 0, :]) / (shift[1] - shift[0])

    # Save velocities and gradients with the corresponding y value in the filename
    np.save(f'velocities-{fixed_y_value:.3f}.npy', veloc)
    np.save(f'gradients-{fixed_y_value:.3f}.npy', grads)